# TN2 — Tầm nhìn của DS-TCN 192 kênh

## Câu hỏi

Cấu hình DS-TCN 192 kênh có tầm nhìn **61 mẫu** trên cửa sổ vào **200 mẫu**. Nó
chỉ thấy 30% cửa sổ, tức 1,2 giây ở tần số 50 Hz — chưa tới một nhịp thở, vốn
dài khoảng 4 giây.

**Mở rộng tầm nhìn tới quanh 200 mẫu có làm điểm khá lên không?**

## Thang thí nghiệm

Giữ nguyên **4 khối**, chỉ đổi bề rộng kernel. Ở cùng số khối, kernel tăng làm
trọng số depthwise tăng, nhưng phần pointwise mới chiếm đa số — nên số tham số
gần như đứng yên:

| kernel | tầm nhìn | phủ được cửa sổ | tham số | so với gốc |
|---:|---:|---:|---:|---:|
| 3 | 61 | 30% | 307.801 | — *(đang chạy ở TN1)* |
| **5** | **121** | **60%** | **310.873** | **+1,0%** |
| **7** | **181** | **90%** | **313.945** | **+2,0%** |
| **9** | **241** | **100%** | **317.017** | **+3,0%** |
| **11** | **301** | **100%** | **320.089** | **+4,0%** |
| **13** | **361** | **100%** | **323.161** | **+5,0%** |

**Tầm nhìn gấp 5,9 lần mà tham số chỉ chênh 5%.** Vì vậy nếu điểm đổi theo
thang này thì gần như chắc là do tầm nhìn, không phải do sức chứa — điều mà
thí nghiệm đổi số khối không tách được.

Năm cấu hình bắc qua mốc 200: hai cái dưới (121, 181), một cái vừa đủ (241), và
hai cái thừa (301, 361). Nếu điểm lên tới 241 rồi phẳng thì "đủ nhìn hết cửa sổ"
là điều kiện đủ. Nếu vẫn lên sau 241 thì tầm nhìn dài hơn cửa sổ vẫn có ích, và
đó là kết quả đáng nói.

`k=3` không chạy lại ở đây vì đã có trong TN1.

## Chạy vòng sàng lọc, không phải vòng kết luận

Mỗi cấu hình chạy **một fold `val_KL`** — train ABCDEF, chấm K và L, 218.088 cửa
sổ — thay vì đủ bốn fold. Năm cấu hình mất khoảng **2,5 giờ** thay vì 10 giờ.

**Hai điều bắt buộc nhớ khi đọc kết quả**, đo trên tám cấu hình TN1 có đủ ba seed:

**1. Điểm một fold không so được với `cv_score` bốn fold.** `val_KL` là fold dễ
nhất; điểm trên nó cao hơn trung bình **+0,040**, và mức chênh không đều, từ
+0,020 tới +0,070 tuỳ cấu hình.

**2. Thứ hạng có thể đảo.** BiLSTM-41 đứng chót trên bốn fold nhưng hạng ba nếu
chỉ nhìn `val_KL`; DS-TCN-64 thì từ hạng sáu xuống chót.

Nên vòng này dùng để **loại**, không dùng để **chọn**. Cấu hình nào ở sát nhau
thì đưa cả cụm sang vòng xác nhận. Chi tiết: `docs/SANG_LOC.md`.

`run_cv.py` **không ghi dòng `TONG`** khi chạy thiếu fold, nên số sàng lọc không
lọt vào bảng `compare_cv`. Muốn xác nhận thì chạy lại **bỏ `--folds`**, fold đã
xong được bỏ qua và dòng `TONG` tự có.

## Mốc để đặt cạnh

So **một fold với một fold**. Điểm `val_KL` seed 0 của các cấu hình TN1:

| | tham số | tầm nhìn | val_KL seed 0 |
|---|---:|---:|---:|
| LSTM-67 | 56.908 | — | 0,8211 |
| LSTM-352 | 1.502.713 | — | 0,8257 |
| DS-TCN-64 | 56.281 | 253 | 0,7723 |
| TCN-64 | 151.513 | 253 | 0,7872 |

Nhóm này ghi vào `runs/tn2_rf/`, tách khỏi `tn2` của thí nghiệm RevIN.

## 1. Chuẩn bị Colab

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn. Cờ `--folds` và `--norm none` chỉ có ở bản mới.

In [ ]:
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

## 2. Kiểm năm bản cài đặt

Mỗi lệnh in tầm nhìn ở mục 7, đối chiếu với bảng đầu notebook. Số tham số phải
ra đúng 310.873 / 313.945 / 317.017 / 320.089 / 323.161.

**Đọc dòng cuối mỗi lệnh trước khi chạy tiếp.** Phải là `TẤT CẢ ĐẠT` — notebook
không tự dừng khi lệnh trượt.

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 7 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 9 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 11 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 13 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

## 3. Chạy năm cấu hình, một fold, một seed

Tên cấu hình: `ds_tcn_c192_k<K>_n4_none_do0.2_dpel_mse_corr0.9_seed0`.

Mỗi lệnh khoảng **30 phút**. Kernel lớn hơn thì tích chập depthwise nặng hơn
chút, nên k=13 lâu hơn k=5 một ít.

**kernel 5 — tầm nhìn 121, 310.873 tham số**

In [ ]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --folds val_KL --seed 0

**kernel 7 — tầm nhìn 181, 313.945 tham số**

In [ ]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 192 \
    --kernel_size 7 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --folds val_KL --seed 0

**kernel 9 — tầm nhìn 241, 317.017 tham số**

In [ ]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 192 \
    --kernel_size 9 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --folds val_KL --seed 0

**kernel 11 — tầm nhìn 301, 320.089 tham số**

In [ ]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 192 \
    --kernel_size 11 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --folds val_KL --seed 0

**kernel 13 — tầm nhìn 361, 323.161 tham số**

In [ ]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 192 \
    --kernel_size 13 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --folds val_KL --seed 0

## 4. Cất kết quả

In [ ]:
!python scripts/save_results.py tn2_rf --out tn2_rf_ds_tcn_c192

## 5. Đọc kết quả

`compare_cv` sẽ **không** hiện gì, vì chạy thiếu fold nên không có dòng `TONG`.
Đó là chủ ý. Điểm từng cấu hình đọc thẳng từ đầu ra ở mục 3, dòng
`fold  0.xxxx` của mỗi lệnh.

Ô dưới in lại năm dòng đó cho gọn.

In [ ]:
import csv, re
RF = {3: 61, 5: 121, 7: 181, 9: 241, 11: 301, 13: 361}
rows = [r for r in csv.DictReader(open("runs/tn2_rf/summary.csv")) if r["fold"] == "val_KL"]
for r in sorted(rows, key=lambda r: int(re.search(r"_k(\d+)_", r["run_id"]).group(1))):
    k = int(re.search(r"_k(\d+)_", r["run_id"]).group(1))
    print("  kernel %-3d tầm nhìn %-4d %8s tham số   %s"
          % (k, RF[k], r["n_params"], r["score_macro"]))

## 6. Đường cong tầm nhìn

Vẽ điểm theo tầm nhìn để thấy nó phẳng ở đâu. Mốc 200 là bề rộng cửa sổ vào.

In [ ]:
import csv, re, matplotlib.pyplot as plt
RF = {3:61, 5:121, 7:181, 9:241, 11:301, 13:361}
d = {RF[int(re.search(r"_k(\d+)_", r["run_id"]).group(1))]: float(r["score_macro"])
     for r in csv.DictReader(open("runs/tn2_rf/summary.csv")) if r["fold"] == "val_KL"}
x = sorted(d); plt.plot(x, [d[i] for i in x], "o-")
plt.axvline(200, ls="--", c="r", label="bề rộng cửa sổ 200")
plt.xlabel("tầm nhìn (mẫu)"); plt.ylabel("điểm val_KL"); plt.legend(); plt.grid(alpha=.3)

## 7. Ngắt phiên

In [ ]:
from google.colab import runtime
runtime.unassign()